# Sampling, Conversion and Quantization
This notebook covers the projects in this folder's README:
1. **Audio sampling and reconstruction**: the Nyquist-Shannon theorem, spectral images and aliasing.
2. **Analog and digital audio conversion**: the stages of an ADC and a DAC, and how real converters differ from the ideal.
3. **Audio quantization effects**: bit depth, quantization noise, quantizer types and dither.

A computer cannot hold a truly continuous signal. Here an "analog" signal is approximated by evaluating it on a dense 200 kHz time grid, which is fine enough to look and behave continuous for audio frequencies.

Background reading: *Audio Engineering Know It All*, chapter 14 (John Watkinson, "Digital Audio Fundamentals") and chapter 15 (Ian Sinclair and Allen Mornington-West, "Representation of Audio Signals").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import multi_sine, spectrum_db

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

ANALOG_RATE = 200_000  # dense grid that stands in for continuous time

def analog_time(duration):
    return np.arange(int(duration * ANALOG_RATE)) / ANALOG_RATE

## 1. Sampling and reconstruction
Sampling at rate $f_s$ keeps only the values $x[n] = x(nT)$, with sample period $T = 1/f_s$. The Nyquist-Shannon theorem says that a signal containing only frequencies below $f_s/2$ (the **Nyquist frequency**) can be rebuilt exactly from its samples using sinc interpolation:

$$x(t) = \sum_n x[n]\,\mathrm{sinc}\!\left(\frac{t - nT}{T}\right)$$

A frequency $f$ above $f_s/2$ produces exactly the same samples as a lower **alias** frequency, so after sampling the two cannot be told apart. The alias is at

$$f_{alias} = \left| f - f_s \cdot \mathrm{round}(f / f_s) \right|$$

Move the frequency past the Nyquist frequency and watch the reconstruction lock onto the alias instead of the original.

In [ ]:
def alias_frequency(f, fs):
    return abs(f - fs * round(f / fs))

def sinc_reconstruct(samples, fs, t):
    """Whittaker-Shannon interpolation of samples taken at n / fs, evaluated at times t."""
    n = np.arange(len(samples))
    return samples @ np.sinc(fs * t[None, :] - n[:, None])

def plot_sampling(f=1000, fs=8000):
    # Sample 15 ms, but plot 4 ms from the middle: sinc interpolation of a
    # finite set of samples is inaccurate near the ends.
    t_start, t_len = 0.005, 0.004
    n = np.arange(int(0.015 * fs))
    samples = np.sin(2 * np.pi * f * n / fs)
    t = t_start + np.arange(int(t_len * ANALOG_RATE)) / ANALOG_RATE

    in_view = (n / fs >= t_start) & (n / fs <= t_start + t_len)
    fig, ax = plt.subplots()
    ax.plot(t * 1000, np.sin(2 * np.pi * f * t), color='0.6', linewidth=2, label=f'original ({f} Hz)')
    ax.plot(t * 1000, sinc_reconstruct(samples, fs, t), 'r--', linewidth=1.5, label='sinc reconstruction')
    ax.stem(n[in_view] / fs * 1000, samples[in_view], basefmt=' ', label=f'samples ({fs} Hz)')
    title = f'f = {f} Hz, fs = {fs} Hz, Nyquist = {fs / 2:.0f} Hz'
    if f > fs / 2:
        title += f'  ->  aliased to {alias_frequency(f, fs):.0f} Hz'
    ax.set_title(title)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.legend(loc='upper right')
    plt.show()

display.display(interactive(
    plot_sampling,
    f=widgets.IntSlider(value=1000, min=100, max=20000, step=100, description='Signal (Hz):'),
    fs=widgets.Dropdown(options=[4000, 8000, 16000, 22050, 44100], value=8000, description='fs (Hz):'),
))

### Why aliasing happens: spectral images
Sampling can be modeled as multiplying $x(t)$ by a train of impulses one sample period apart:

$$x_s(t) = x(t) \cdot \sum_{n} \delta(t - nT)$$

Multiplication in time is convolution in frequency, and the spectrum of an impulse train is itself an impulse train at multiples of $f_s$. So the sampled spectrum is the original spectrum repeated around every multiple of the sample rate:

$$X_s(f) = \frac{1}{T} \sum_{k} X(f - k f_s)$$

Each copy is called an **image**. If the signal has content above $f_s/2$, the image centered on $f_s$ reaches down into the baseband ($0$ to $f_s/2$, shaded green below). That overlap is aliasing. Once it has happened, no filter can separate the alias from genuine content at the same frequency.

Below, signals are "sampled" at 8 kHz on the dense grid by keeping every 25th point and setting the rest to zero, which is the impulse-train model above.

In [ ]:
fs = 8000
hold = ANALOG_RATE // fs
t = analog_time(0.1)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, tones, title in [
    (axes[0], [500, 1500, 3000], 'Tones at 0.5, 1.5 and 3 kHz: images around every multiple of 8 kHz, baseband clean'),
    (axes[1], [500, 1500, 5500], 'Tone at 5.5 kHz is above fs/2: its image at 8 - 5.5 = 2.5 kHz lands in the baseband'),
]:
    x = multi_sine(tones, 0.1, ANALOG_RATE) / len(tones)
    impulses = np.zeros_like(x)
    impulses[::hold] = x[::hold]
    f, mag = spectrum_db(impulses, ANALOG_RATE)
    ax.plot(f / 1000, mag)
    ax.axvspan(0, fs / 2 / 1000, color='green', alpha=0.1)
    for k in range(1, 5):
        ax.axvline(k * fs / 1000, color='k', linestyle=':', linewidth=1)
    ax.set_xlim(0, 34)
    ax.set_ylim(-80, 5)
    ax.set_ylabel('dB')
    ax.set_title(title)
axes[1].set_xlabel('Frequency (kHz)   (dotted lines: multiples of fs)')
plt.tight_layout()
plt.show()

### Hearing aliasing
A sweep from 100 Hz to 16 kHz is produced two ways, both ending at an 8 kHz sample rate:
- **Naive:** the sweep is computed directly at 8 kHz. Everything above 4 kHz aliases, so the pitch rises to 4 kHz and then *falls* back down as it folds.
- **Anti-aliased:** the sweep is generated at 48 kHz, low-pass filtered below 4 kHz, and then every 6th sample is kept (`scipy.signal.resample_poly` does both steps). The sweep fades out once it passes 4 kHz.

In [ ]:
duration, f0, f1 = 4.0, 100, 16000
fs_hi, fs_lo = 48000, 8000

t_lo = np.arange(int(duration * fs_lo)) / fs_lo
naive = signal.chirp(t_lo, f0, duration, f1)

t_hi = np.arange(int(duration * fs_hi)) / fs_hi
anti_aliased = signal.resample_poly(signal.chirp(t_hi, f0, duration, f1), up=1, down=fs_hi // fs_lo)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, x, name in [(axes[0], naive, 'Naive sampling at 8 kHz'),
                    (axes[1], anti_aliased, 'Anti-alias filter, then 8 kHz')]:
    f, tt, Sxx = signal.spectrogram(x, fs_lo, nperseg=256)
    S = 10 * np.log10(Sxx + 1e-12)
    ax.pcolormesh(tt, f, S, shading='auto', vmin=S.max() - 80)
    ax.set_title(name)
    ax.set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (Hz)')
plt.show()

print('Naive:')
display.display(display.Audio(naive, rate=fs_lo))
print('Anti-aliased:')
display.display(display.Audio(anti_aliased, rate=fs_lo))

## 2. Analog and digital audio conversion
An audio interface converts between analog and digital in these stages:

| Stage | Converter | What it does |
|---|---|---|
| 1. Anti-alias filter | ADC | Analog low-pass filter that removes content above $f_s/2$ before sampling |
| 2. Sample-and-hold | ADC | Measures the voltage every $T$ seconds |
| 3. Quantizer | ADC | Rounds each sample to one of $2^B$ levels ($B$ = bit depth) |
| 4. Zero-order hold | DAC | Holds each sample value as a voltage for one period $T$, giving a staircase |
| 5. Reconstruction filter | DAC | Analog low-pass filter that smooths the staircase |

The example uses a deliberately low sample rate (8 kHz) and bit depth (4 bits) so each stage is visible. The input is a 1 kHz tone plus a weaker 5.5 kHz tone. 5.5 kHz is above the 4 kHz Nyquist frequency, so without stage 1 it would alias to 2.5 kHz.

The analog filters are approximated by 8th-order Butterworth filters applied on the dense grid. `sosfiltfilt` runs each filter forwards and backwards, which removes its delay so the stages line up in the plots. A real analog filter cannot do this.

In [ ]:
def quantize(x, bits, full_scale=1.0):
    """Uniform mid-tread quantizer with 2**bits levels over [-full_scale, full_scale)."""
    step = 2 * full_scale / 2**bits
    return np.clip(np.round(x / step) * step, -full_scale, full_scale - step)

fs, bits = 8000, 4
hold = ANALOG_RATE // fs                     # dense-grid points per sample period
t = analog_time(0.05)
analog_in = 0.7 * np.sin(2 * np.pi * 1000 * t) + 0.25 * np.sin(2 * np.pi * 5500 * t)

lowpass = signal.butter(8, 3400, fs=ANALOG_RATE, output='sos')
filtered = signal.sosfiltfilt(lowpass, analog_in)        # 1. anti-alias filter
samples = filtered[::hold]                               # 2. sample-and-hold
digital = quantize(samples, bits)                        # 3. quantizer
staircase = np.repeat(digital, hold)                     # 4. zero-order hold
analog_out = signal.sosfiltfilt(lowpass, staircase)      # 5. reconstruction filter

view = (t >= 0.02) & (t < 0.025)
n_view = (np.arange(len(samples)) / fs >= 0.02) & (np.arange(len(samples)) / fs < 0.025)
t_n = np.arange(len(samples))[n_view] / fs * 1000

fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
axes[0].plot(t[view] * 1000, analog_in[view])
axes[0].set_title('Analog input: 1 kHz + 5.5 kHz')
axes[1].plot(t[view] * 1000, filtered[view])
axes[1].set_title('1. After anti-alias filter (5.5 kHz removed)')
axes[2].plot(t[view] * 1000, filtered[view], color='0.7')
axes[2].stem(t_n, samples[n_view], basefmt=' ')
axes[2].set_title(f'2. Sampled at {fs} Hz')
axes[3].stem(t_n, digital[n_view], basefmt=' ')
for level in np.arange(-1, 1, 2 / 2**bits):
    axes[3].axhline(level, color='0.85', linewidth=0.8, zorder=0)
axes[3].set_title(f'3. Quantized to {bits} bits ({2**bits} levels, grey lines)')
axes[4].plot(t[view] * 1000, staircase[view], label='4. zero-order hold')
axes[4].plot(t[view] * 1000, analog_out[view], 'r', label='5. after reconstruction filter')
axes[4].set_title('DAC output')
axes[4].legend(loc='upper right')
axes[4].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

The staircase from the zero-order hold contains copies (**images**) of the 1 kHz tone around every multiple of $f_s$: at 7 and 9 kHz, 15 and 17 kHz, and so on. Their level follows a $\mathrm{sinc}$ envelope. The reconstruction filter removes them. The right-hand plot repeats the ADC without the anti-alias filter: the 5.5 kHz tone appears at its 2.5 kHz alias in the digital signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for x, label in [(staircase, 'zero-order hold output'), (analog_out, 'after reconstruction filter')]:
    f, mag = spectrum_db(x, ANALOG_RATE)
    axes[0].plot(f / 1000, mag, label=label)
axes[0].set_xlim(0, 40)
axes[0].set_ylim(-100, 5)
axes[0].set_title('DAC output spectrum: images around multiples of 8 kHz')
axes[0].set_xlabel('Frequency (kHz)')
axes[0].set_ylabel('dB')
axes[0].legend()

for x, label in [(quantize(analog_in[::hold], bits), 'no anti-alias filter'), (digital, 'with anti-alias filter')]:
    f, mag = spectrum_db(x, fs)
    axes[1].plot(f / 1000, mag, label=label)
axes[1].axvline(2.5, color='r', linestyle=':', label='2.5 kHz alias of 5.5 kHz')
axes[1].set_ylim(-60, 5)
axes[1].set_title(f'Digital signal spectrum ({fs} Hz, {bits} bits)')
axes[1].set_xlabel('Frequency (kHz)')
axes[1].legend()
plt.tight_layout()
plt.show()

### Ideal sampling vs real converters
| | Ideal | Real converters |
|---|---|---|
| Signal bandwidth | Nothing above $f_s/2$ | Real signals and noise extend above $f_s/2$, so an anti-alias filter is always needed |
| Anti-alias filter | Brick wall: full attenuation from $f_s/2$, no phase shift | A finite-order filter needs a transition band. Modern converters oversample and do most of the filtering digitally |
| Sampling pulse | Zero-width impulses | The sample-and-hold measures over a finite **aperture** time $\tau$ |
| Observation | Infinitely long signal | A finite recording, which causes spectral leakage (see `02_Frequency_Domain_Analysis`) |
| Reconstruction | Sinc interpolation | Zero-order hold, then an analog reconstruction filter |

**Aperture effect.** Holding or averaging each sample over a time $\tau$ multiplies the spectrum by $\mathrm{sinc}(f\tau) = \sin(\pi f \tau)/(\pi f \tau)$. A zero-order hold DAC has $\tau = T$, so the top of the band droops by up to 3.9 dB at $f_s/2$. This is the sinc envelope on the images above. DACs compensate with a small high-frequency boost, or by shortening $\tau$.

In [ ]:
fs = 8000
for f in [1000, 2000, 3000, 4000]:
    print(f'Zero-order hold droop at {f} Hz (fs = {fs} Hz): {20 * np.log10(np.sinc(f / fs)):6.2f} dB')

## 3. Audio quantization effects
A $B$-bit quantizer has step size $\Delta = 2/2^B$ over the range $[-1, 1)$. Rounding to the nearest level leaves an error $e[n]$ between $-\Delta/2$ and $+\Delta/2$. When the signal is large and busy, this error behaves like white noise, spread evenly over that range, with power

$$\sigma_e^2 = \frac{\Delta^2}{12}$$

A full-scale sine has power $1/2$, which gives the signal-to-quantization-noise ratio

$$\mathrm{SQNR} = 10\log_{10}\frac{1/2}{\Delta^2/12} \approx 6.02\,B + 1.76 \ \text{dB}$$

Each extra bit adds about 6 dB. 16-bit audio gives about 98 dB.

### Mid-tread and mid-riser quantizers
- **Mid-tread** quantizers (used above) have a level at exactly zero. Silence stays silent. Integer PCM formats are mid-tread, with one more negative level than positive (16-bit runs from -32768 to +32767).
- **Mid-riser** quantizers have levels at $\pm\Delta/2, \pm 3\Delta/2, \dots$ and no zero. Any tiny input, even background noise far below one step, comes out as a full $\pm\Delta/2$ signal.

In [ ]:
def quantize_mid_riser(x, bits, full_scale=1.0):
    """Uniform mid-riser quantizer: 2**bits levels at odd multiples of step / 2."""
    step = 2 * full_scale / 2**bits
    return np.clip((np.floor(x / step) + 0.5) * step, -full_scale + step / 2, full_scale - step / 2)

bits = 3
x_ramp = np.linspace(-1, 1, 2001)
step = 2 / 2**bits
rng = np.random.default_rng(0)
quiet_noise = rng.uniform(-0.3, 0.3, 200) * step   # input noise well below one step

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(x_ramp, quantize(x_ramp, bits), label='mid-tread')
axes[0].plot(x_ramp, quantize_mid_riser(x_ramp, bits), label='mid-riser')
axes[0].plot(x_ramp, x_ramp, 'k:', linewidth=1, label='input')
axes[0].set_title(f'{bits}-bit transfer curves ({2**bits} levels each)')
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')
axes[0].legend()
axes[1].plot(quiet_noise, 'k', linewidth=1, label='input (noise below one step)')
axes[1].plot(quantize(quiet_noise, bits), label='mid-tread output')
axes[1].plot(quantize_mid_riser(quiet_noise, bits), label='mid-riser output')
axes[1].set_title('Near-silent input')
axes[1].set_xlabel('Sample')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.show()
print('Distinct output levels, mid-tread:', len(np.unique(quantize(x_ramp, bits))),
      ' mid-riser:', len(np.unique(quantize_mid_riser(x_ramp, bits))))

### Dither, and SQNR with dither
Adding a small random signal, **dither**, before quantizing breaks the link between the signal and the error (explained in "Why dither?" below). Two common kinds:
- **RPDF** (rectangular PDF): one uniform random value spanning $\pm\Delta/2$. Variance $\Delta^2/12$.
- **TPDF** (triangular PDF): the sum of two such values, spanning $\pm\Delta$. Variance $\Delta^2/6$.

The total noise is the dither plus the quantization error. With TPDF that is $\Delta^2/6 + \Delta^2/12 = \Delta^2/4$, three times (4.77 dB) more than without dither:

$$\mathrm{SQNR}_{\text{TPDF}} \approx 6.02\,B - 3.01 \ \text{dB}$$

The plot measures both. The test sine is at half full scale (-6 dBFS) so dither cannot push it into clipping, and the noise is expressed relative to a full-scale sine.

In [ ]:
def rpdf_dither(n_samples, bits, rng):
    """Rectangular-PDF dither spanning +/- half a quantization step."""
    return rng.uniform(-0.5, 0.5, n_samples) * (2 / 2**bits)

def tpdf_dither(n_samples, bits, rng):
    """Triangular-PDF dither spanning +/- 1 quantization step."""
    return (rng.uniform(-0.5, 0.5, n_samples) + rng.uniform(-0.5, 0.5, n_samples)) * (2 / 2**bits)

def sqnr_full_scale_db(x, xq):
    """Quantization noise power expressed relative to a full-scale sine (power 1/2)."""
    return 10 * np.log10(0.5 / np.mean((xq - x)**2))

fs = 44100
n = np.arange(fs)
# 997 Hz shares no common factor with 44100, so the error pattern does not repeat every few samples
test_sine = 0.5 * np.sin(2 * np.pi * 997 * n / fs)

bit_depths = np.arange(3, 17)
plain = [sqnr_full_scale_db(test_sine, quantize(test_sine, b)) for b in bit_depths]
dithered = [sqnr_full_scale_db(test_sine, quantize(test_sine + tpdf_dither(len(n), b, rng), b)) for b in bit_depths]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(bit_depths, plain, 'o', label='measured, no dither')
ax.plot(bit_depths, 6.02 * bit_depths + 1.76, '-', label='6.02 B + 1.76 dB')
ax.plot(bit_depths, dithered, 's', label='measured, TPDF dither')
ax.plot(bit_depths, 6.02 * bit_depths - 3.01, '--', label='6.02 B - 3.01 dB')
ax.set_xlabel('Bit depth B')
ax.set_ylabel('SQNR (dB)')
ax.set_title('Signal-to-quantization-noise ratio')
ax.legend()
plt.show()

### Listening to bit depth
The test signal is a C major chord that decays over 3 seconds, like a struck piano chord. Quantization error is most audible at the quiet end, where the signal spans only a few quantization levels. Lower the bit depth and listen to the tail of the note, then try the two kinds of dither.

In [ ]:
t = np.arange(3 * fs) / fs
chord = multi_sine([261.63, 329.63, 392.00, 523.25], 3.0, fs, amplitudes=[0.35, 0.3, 0.25, 0.1]) * np.exp(-1.5 * t)
DITHERS = {'none': lambda n_samples, bits, rng: 0, 'RPDF': rpdf_dither, 'TPDF': tpdf_dither}

def plot_quantization(bits=6, dither='none'):
    xq = quantize(chord + DITHERS[dither](len(chord), bits, rng), bits)
    tail = (t >= 2.5) & (t < 2.52)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(t[tail] * 1000, chord[tail], label='original')
    axes[0].plot(t[tail] * 1000, xq[tail], label='quantized')
    axes[0].set_title('Waveform at 2.5 s (quiet tail)')
    axes[0].set_xlabel('Time (ms)')
    axes[0].legend()
    axes[1].plot(t[tail] * 1000, (xq - chord)[tail])
    axes[1].set_title('Total error (quantization + dither)')
    axes[1].set_xlabel('Time (ms)')
    f, mag = spectrum_db(xq[t >= 2], fs)
    axes[2].plot(f, mag)
    axes[2].set_xlim(0, 5000)
    axes[2].set_ylim(-100, 5)
    axes[2].set_title('Spectrum of the last second')
    axes[2].set_xlabel('Frequency (Hz)')
    plt.tight_layout()
    plt.show()
    snr = 10 * np.log10(np.mean(chord**2) / np.mean((xq - chord)**2))
    print(f'{bits} bits, dither {dither}: SNR = {snr:.1f} dB')
    display.display(display.Audio(xq, rate=fs))

display.display(interactive(
    plot_quantization,
    bits=widgets.IntSlider(value=6, min=1, max=16, description='Bits:'),
    dither=widgets.ToggleButtons(options=list(DITHERS), description='Dither:'),
))

### Why dither?
For a quiet signal the quantization error is no longer noise-like. It is a fixed function of the signal, so it repeats with the signal's period and shows up as **harmonic distortion**. Dither makes the error random, and the kind of dither decides how random:

| | Mean error | Error power | Result |
|---|---|---|---|
| No dither | Depends on the signal value | Depends on the signal value | Distortion |
| RPDF | 0 for every signal value | Still depends on the signal value | No distortion, but the noise level rises and falls with the signal (**noise modulation**) |
| TPDF | 0 for every signal value | Constant, $\Delta^2/4$ | No distortion and a steady noise floor |

The cell below checks this directly. It holds the input at each value across one quantization step, adds many independent dither values, and measures the mean and power of the total error. Units are quantization steps.

In [ ]:
v = np.linspace(0, 1, 201)                     # constant input values across one step
d_rpdf = rng.uniform(-0.5, 0.5, 20000)
d_tpdf = rng.uniform(-0.5, 0.5, 20000) + rng.uniform(-0.5, 0.5, 20000)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for d, name in [(np.zeros(1), 'no dither'), (d_rpdf, 'RPDF'), (d_tpdf, 'TPDF')]:
    error = np.round(v[:, None] + d[None, :]) - v[:, None]   # total error for every (input, dither) pair
    axes[0].plot(v, error.mean(axis=1), label=name)
    axes[1].plot(v, (error**2).mean(axis=1), label=f'{name} (average {np.mean(error**2):.3f})')
axes[0].set_title('Mean error vs input value')
axes[1].set_title('Error power vs input value')
for value, label in [(1 / 12, '1/12'), (1 / 6, '1/6'), (1 / 4, '1/4')]:
    axes[1].axhline(value, color='0.6', linestyle=':', linewidth=1)
    axes[1].text(1.01, value, label, va='center')
for ax in axes:
    ax.set_xlabel('Input value (quantization steps)')
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In audio, a 997 Hz sine with an amplitude of only 1.5 quantization steps is quantized to 8 bits. The power spectral densities are estimated with Welch's method (`scipy.signal.welch`), which averages the spectra of many overlapping segments to give a smooth noise floor. Without dither the error shows up as harmonics at 3x, 5x, 7x ... 997 Hz. Both dithers replace them with a flat noise floor. TPDF's floor is slightly higher, the price of also removing noise modulation.

In [ ]:
bits = 8
quiet_sine = 1.5 * (2 / 2**bits) * np.sin(2 * np.pi * 997 * n / fs)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, name in zip(axes, DITHERS):
    xq = quantize(quiet_sine + DITHERS[name](len(quiet_sine), bits, rng), bits)
    f, psd = signal.welch(xq, fs, nperseg=8192)
    ax.plot(f, 10 * np.log10(psd + 1e-20))
    ax.set_xlim(0, 12000)
    ax.set_ylim(-130, -40)
    ax.set_title(f'{name}: SNR {10 * np.log10(np.mean(quiet_sine**2) / np.mean((xq - quiet_sine)**2)):.1f} dB')
    ax.set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('PSD (dB re full scale$^2$/Hz)')
plt.tight_layout()
plt.show()